# Copy-cat Bot (Google Colab)

Learns the style of one speaker from their speeches and writes new text in that style.

**How to use:** run the cells from top to bottom (Runtime > Run all works once your data is in place).

| Part | What | Needs GPU? |
|---|---|---|
| 1 | Setup, data, cleaning | No |
| 2 | Markov chain bot (the main project) | No. Builds in seconds on CPU |
| 3 | Optional: LSTM, compared against Markov | Yes. Runtime > Change runtime type > T4 GPU |

All generated text is **AI-generated**. It is not a real quote from the speaker.

## Part 1: Setup and data

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path("/content/copycat-bot")
for p in ["src", "data/raw", "data/clean", "results"]:
    (ROOT / p).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project folder:", ROOT)

### Load the speeches
Choose one:
- `"drive"` (recommended): keep your `.txt` transcripts in a Google Drive folder. Colab deletes files in `/content` when the session resets, so Drive saves you from uploading again.
- `"upload"`: pick files from your computer each session.

Set `DRIVE_FOLDER` to the Drive folder that holds your `.txt` files.

In [ ]:
DATA_SOURCE = "drive"        # "drive" or "upload"
DRIVE_FOLDER = "/content/drive/MyDrive/copycat-bot/raw"

if DATA_SOURCE == "upload":
    from google.colab import files
    for name, content in files.upload().items():
        (ROOT / "data/raw" / name).write_bytes(content)
else:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    found = list(Path(DRIVE_FOLDER).glob("*.txt"))
    assert found, f"No .txt files in {DRIVE_FOLDER}. Check the folder path."
    for f in found:
        shutil.copy(f, ROOT / "data/raw" / f.name)

print(len(list((ROOT / "data/raw").glob("*.txt"))), "file(s) in data/raw")

### Project code
These three cells write the same `src/` files as the project structure in the brief.

In [ ]:
%%writefile src/clean.py
"""Step 2: clean raw transcripts in data/raw/ and save to data/clean/."""
import argparse
import html
import re
import unicodedata
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent

STAGE_WORDS = (r"applause|laughter|laughs|cheers|cheering|music|audience|interruption|"
               r"inaudible|pause|crosstalk|slogans|video|chanting|clapping|translation")
RE_TAGS = re.compile(r"<[^>]+>")
RE_URLS = re.compile(r"https?://\S+|www\.\S+")
RE_SQUARE = re.compile(r"\[[^\]]{0,100}\]")                       # [applause], [inaudible]
RE_PAREN_STAGE = re.compile(rf"\((?=[^)]*(?:{STAGE_WORDS}))[^)]{{0,100}}\)", re.I)
MONTHS = r"(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)"
RE_DATES = re.compile(
    rf"\b\d{{1,2}}(?:st|nd|rd|th)?\s+{MONTHS}\.?,?\s+\d{{4}}\b"   # 12 March 2019
    rf"|\b{MONTHS}\.?\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+\d{{4}}\b"  # March 12, 2019
    r"|\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"                         # 12/03/2019
    r"|\b\d{4}-\d{2}-\d{2}\b",                                    # 2019-03-12
    re.I,
)
QUOTES = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
          "\u2013": "-", "\u2014": " - ", "\u2026": "..."}


def read_text(path: Path) -> str:
    """Fix encoding problems: try UTF-8 first, then Windows-1252, then Latin-1."""
    raw = path.read_bytes()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace")


def clean_text(text: str) -> str:
    text = html.unescape(text)
    text = RE_TAGS.sub(" ", text)
    text = unicodedata.normalize("NFKC", text)
    for bad, good in QUOTES.items():
        text = text.replace(bad, good)
    text = RE_URLS.sub(" ", text)
    text = RE_SQUARE.sub(" ", text)
    text = RE_PAREN_STAGE.sub(" ", text)
    text = RE_DATES.sub(" ", text)
    text = "".join(ch for ch in text if ch.isprintable() or ch in "\n\t")
    return re.sub(r"\s+", " ", text).strip()


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--raw", default=ROOT / "data" / "raw")
    ap.add_argument("--out", default=ROOT / "data" / "clean")
    args = ap.parse_args()

    raw_dir, out_dir = Path(args.raw), Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    files = sorted(raw_dir.glob("*.txt"))
    if not files:
        raise SystemExit(f"No .txt files found in {raw_dir}. Put the raw transcripts there first.")

    total = 0
    for f in files:
        cleaned = clean_text(read_text(f))
        (out_dir / f.name).write_text(cleaned, encoding="utf-8")
        total += len(cleaned.split())
    print(f"Cleaned {len(files)} file(s), about {total:,} words -> {out_dir}")
    if total < 20_000:
        print("WARNING: under ~20,000 words. The bot will mostly repeat the originals.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/model.py
"""Steps 3 and 4: split text into tokens and build the Markov chain model."""
import re
from collections import Counter, defaultdict
from pathlib import Path

# A token is a word (apostrophes/hyphens allowed inside) or one punctuation mark.
TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[.!?,;:]")
SENTENCE_END = {".", "!", "?"}


def tokenize(text):
    return TOKEN_RE.findall(text)


def load_docs(clean_dir):
    """Return one token list per cleaned file."""
    files = sorted(Path(clean_dir).glob("*.txt"))
    if not files:
        raise SystemExit(f"No cleaned files in {clean_dir}. Run: python src/clean.py")
    return [tokenize(f.read_text(encoding="utf-8")) for f in files]


def build_model(docs, order=2):
    """
    model[(w1, ..., w_order)] = Counter({next_word: times_seen})
    States are lowercase so "My dear" and "my dear" share counts.
    Next words keep their original case so the output looks natural.
    Also returns `starts`: states that began a sentence (repeats = more likely).
    """
    model = defaultdict(Counter)
    starts = []
    for tokens in docs:
        for i in range(len(tokens) - order):
            state = tuple(t.lower() for t in tokens[i:i + order])
            model[state][tokens[i + order]] += 1
            begins_sentence = i == 0 or tokens[i - 1] in SENTENCE_END
            if begins_sentence and state[0] not in SENTENCE_END | {",", ";", ":"}:
                starts.append(state)
    return model, starts


def detokenize(tokens):
    """Turn tokens back into readable text."""
    out, capitalize = [], True
    for tok in tokens:
        if tok in {".", "!", "?", ",", ";", ":"}:
            if out:
                out[-1] += tok
            else:
                out.append(tok)
            capitalize = tok in SENTENCE_END
            continue
        if capitalize:
            tok = tok[:1].upper() + tok[1:]
            capitalize = False
        out.append(tok)
    return " ".join(out)

In [ ]:
%%writefile src/generate.py
"""Steps 5 and 6: generate new text and check it is not just copying."""
import argparse
import random
from pathlib import Path

from model import SENTENCE_END, build_model, detokenize, load_docs, tokenize

ROOT = Path(__file__).resolve().parent.parent
LABEL = "[AI-generated text. Not a real quote from the speaker.]"


def pick_start(model, starts, order, start_text, rng):
    """Return the starting tokens."""
    if not start_text:
        return list(rng.choice(starts))
    toks = tokenize(start_text)
    low = [t.lower() for t in toks]
    if not low:
        raise SystemExit("--start has no usable words.")
    if len(low) >= order:
        if tuple(low[-order:]) not in model:
            raise SystemExit(f"The phrase {' '.join(low[-order:])!r} never appears in the data. Try another start.")
        return toks
    # Start shorter than the model memory: pick a state that begins with it.
    matches = [s for s in model if list(s[:len(low)]) == low]
    if not matches:
        raise SystemExit(f"The word(s) {' '.join(low)!r} never appear in the data. Try another start.")
    weights = [sum(model[s].values()) for s in matches]
    state = rng.choices(matches, weights=weights)[0]
    return toks + list(state[len(low):])


def generate(model, starts, order, start_text, length, rng, extra=40):
    """Walk the chain. After `length` tokens, keep going until a sentence ends (max `extra` more)."""
    out = pick_start(model, starts, order, start_text, rng)
    while True:
        nxt = model.get(tuple(t.lower() for t in out[-order:]))
        if not nxt:  # dead end (only happens at the very end of a source text)
            if len(out) >= length:
                break
            if out[-1] not in SENTENCE_END:
                out.append(".")
            out += list(rng.choice(starts))
            continue
        words, counts = zip(*nxt.items())
        out.append(rng.choices(words, weights=counts)[0])
        if len(out) >= length and out[-1] in SENTENCE_END:
            break
        if len(out) >= length + extra:
            break
    return out


def build_index(docs, n=8):
    """Set of fingerprints of every n-token window in the source speeches."""
    index = set()
    for d in docs:
        low = [t.lower() for t in d]
        for i in range(len(low) - n + 1):
            index.add(hash(tuple(low[i:i + n])))
    return index


def copy_stats(out_tokens, index, n=8):
    """Return (percent of n-token windows found verbatim in the source, longest copied run in tokens)."""
    low = [t.lower() for t in out_tokens]
    flags = [hash(tuple(low[i:i + n])) in index for i in range(len(low) - n + 1)]
    if not flags:
        return 0.0, 0
    best = run = 0
    for f in flags:
        run = run + 1 if f else 0
        best = max(best, run)
    return 100 * sum(flags) / len(flags), (n + best - 1 if best else 0)


def copy_report(out_tokens, docs, n=8):
    """Step 6: how much of the output is a word-for-word copy of the source?"""
    if len(out_tokens) < n:
        return f"Output shorter than {n} tokens; nothing to check."
    pct, longest = copy_stats(out_tokens, build_index(docs, n), n)
    longest = f"{longest} tokens" if longest else f"under {n} tokens"
    verdict = "OK" if pct < 20 else "HIGH: mostly copying. Need more data or a lower --order."
    return (f"Copy check ({n}-token windows found verbatim in source): {pct:.0f}%\n"
            f"Longest verbatim run: {longest}\nVerdict: {verdict}")


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--start", default="", help='starting word(s), e.g. "my dear"')
    ap.add_argument("--length", type=int, default=100, help="target length in tokens (words + punctuation)")
    ap.add_argument("--order", type=int, default=2, help="how many previous words the model looks at (1-3)")
    ap.add_argument("--seed", type=int, default=None, help="fix the random seed to repeat a result")
    ap.add_argument("--clean-dir", default=ROOT / "data" / "clean")
    ap.add_argument("--no-check", action="store_true", help="skip the copy check")
    args = ap.parse_args()

    docs = load_docs(args.clean_dir)
    total = sum(len(d) for d in docs)
    model, starts = build_model(docs, args.order)
    print(f"Model: {total:,} tokens, {len(model):,} states, order={args.order}")
    if total < 20_000:
        print("WARNING: small dataset. Expect near word-for-word copies.")

    rng = random.Random(args.seed)
    tokens = generate(model, starts, args.order, args.start, args.length, rng)
    print("\n" + detokenize(tokens) + "\n")
    print(LABEL)
    if not args.no_check:
        print("\n" + copy_report(tokens, docs))


if __name__ == "__main__":
    main()

### Step 2: Clean the text

In [ ]:
!python src/clean.py

## Part 2: Markov chain bot
Steps 3 to 6: tokens, model, generate, check.

In [ ]:
import random
from model import load_docs, build_model, detokenize
from generate import generate, copy_report, build_index, copy_stats, LABEL

docs = load_docs(ROOT / "data/clean")
total = sum(len(d) for d in docs)
print(f"{total:,} tokens loaded")
if total < 20_000:
    print("WARNING: small dataset. Expect near word-for-word copies.")

In [ ]:
START = "my dear"   # must appear in your data; use "" for a random sentence start
ORDER = 2           # how many previous words the model looks at (1 to 3)
LENGTH = 100        # tokens (words + punctuation); stops at the next sentence end
SEED = 42

model, starts = build_model(docs, ORDER)
tokens = generate(model, starts, ORDER, START, LENGTH, random.Random(SEED))
markov_text = detokenize(tokens)
print(markov_text, "\n")
print(LABEL, "\n")
print(copy_report(tokens, docs))

### Experiment: does a longer memory help?
Same 10 seeds for every order, so the comparison is fair. A **higher copy %** means the bot is quoting the speeches instead of writing new text. Put this table in your report.

In [ ]:
N = 8
index = build_index(docs, N)
rows = []
for order in (1, 2, 3):
    m, s = build_model(docs, order)
    stats = [copy_stats(generate(m, s, order, "", LENGTH, random.Random(seed)), index, N) for seed in range(10)]
    pcts = [p for p, _ in stats]
    rows.append((order, len(m), sum(pcts) / len(pcts), max(l for _, l in stats)))

print(f"{'order':>5} | {'states':>9} | {'avg copy %':>10} | {'longest copied run':>18}")
for o, st, avg, lg in rows:
    print(f"{o:>5} | {st:>9,} | {avg:>9.1f}% | {lg:>13} tokens")

## Part 3 (optional): LSTM on GPU
This is "Possible improvement #2". Switch to a GPU first: **Runtime > Change runtime type > T4 GPU**, then run this part.

It trains a small word-level LSTM on the same cleaned text. 5% of the text is held back to catch memorizing: if the validation loss starts rising while training continues, the model is learning the speeches by heart. The code keeps the best checkpoint.

Be realistic: with under about 100,000 words, expect the LSTM to be no better than the Markov bot, and possibly worse.

In [ ]:
import torch, torch.nn as nn
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)
if device == "cpu":
    print("No GPU found. Runtime > Change runtime type > T4 GPU (it will still run, slowly).")

VOCAB_SIZE, SEQ_LEN, BATCH, STEPS = 8000, 20, 128, 3000

flat = [t.lower() for d in docs for t in d]
counts = Counter(flat)
itos = ["<unk>"] + [w for w, _ in counts.most_common(VOCAB_SIZE - 1)]
stoi = {w: i for i, w in enumerate(itos)}
ids = torch.tensor([stoi.get(t, 0) for t in flat], dtype=torch.long)

split = int(len(ids) * 0.95)
train_ids, val_ids = ids[:split], ids[split:]
assert len(val_ids) > SEQ_LEN + 2, "Not enough data for an LSTM. Add more speeches."
print(f"vocab={len(itos):,}  train tokens={len(train_ids):,}  validation tokens={len(val_ids):,}")

In [ ]:
class WordLSTM(nn.Module):
    def __init__(self, vocab, emb=128, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.lstm = nn.LSTM(emb, hidden, num_layers=2, batch_first=True, dropout=0.3)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.emb(x), hidden)
        return self.fc(self.drop(out)), hidden


def get_batch(data, batch):
    ix = torch.randint(0, len(data) - SEQ_LEN - 1, (batch,))
    x = torch.stack([data[i:i + SEQ_LEN] for i in ix])
    y = torch.stack([data[i + 1:i + SEQ_LEN + 1] for i in ix])
    return x.to(device), y.to(device)


net = WordLSTM(len(itos)).to(device)
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()
V = len(itos)
best_val, best_state = float("inf"), None

for step in range(1, STEPS + 1):
    net.train()
    x, y = get_batch(train_ids, BATCH)
    logits, _ = net(x)
    loss = loss_fn(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    opt.step()

    if step % 250 == 0:
        net.eval()
        with torch.no_grad():
            vx, vy = get_batch(val_ids, 256)
            val_loss = loss_fn(net(vx)[0].reshape(-1, V), vy.reshape(-1)).item()
        flag = ""
        if val_loss < best_val:
            best_val, flag = val_loss, "  <- best"
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        print(f"step {step:>5} | train loss {loss.item():.3f} | validation loss {val_loss:.3f}{flag}")

net.load_state_dict(best_state)
print("Loaded best checkpoint, validation loss", round(best_val, 3))

In [ ]:
from model import SENTENCE_END, tokenize

def lstm_generate(start_text, length=100, temperature=0.8, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    net.eval()
    out = [t.lower() for t in tokenize(start_text)] or ["my"]
    x = torch.tensor([[stoi.get(t, 0) for t in out]], device=device)
    with torch.no_grad():
        logits, hidden = net(x)
        for _ in range(length + 40):
            scores = logits[0, -1] / temperature
            scores[0] = -float("inf")                      # never output <unk>
            nxt = torch.multinomial(torch.softmax(scores, dim=-1), 1).item()
            out.append(itos[nxt])
            if len(out) >= length and out[-1] in SENTENCE_END:
                break
            logits, hidden = net(torch.tensor([[nxt]], device=device), hidden)
    return out

lstm_tokens = lstm_generate(START, LENGTH, temperature=0.8, seed=SEED)
lstm_text = detokenize(lstm_tokens)
print(lstm_text, "\n")
print(LABEL, "\n")
print(copy_report(lstm_tokens, docs))

### Markov vs LSTM, same copy check on 10 samples each

In [ ]:
lstm_stats = [copy_stats(lstm_generate(START, LENGTH, 0.8, seed=s), index, N) for s in range(10)]
mk_m, mk_s = build_model(docs, 2)
mk_stats = [copy_stats(generate(mk_m, mk_s, 2, START, LENGTH, random.Random(s)), index, N) for s in range(10)]

def avg(stats): return sum(p for p, _ in stats) / len(stats)
print(f"Markov (order 2): avg copy {avg(mk_stats):.1f}%   | LSTM: avg copy {avg(lstm_stats):.1f}%")
print(f"LSTM best validation loss: {best_val:.3f}")
print("Lower copy % is not automatically better: also read the text. Nonsense has 0% copying.")

## Save your results

In [ ]:
out = ROOT / "results" / "output.txt"
parts = [f"MARKOV (order {ORDER}, seed {SEED})\n{markov_text}\n\n{copy_report(tokens, docs)}"]
if "lstm_text" in globals():
    parts.append(f"LSTM (temperature 0.8, seed {SEED})\n{lstm_text}\n\n{copy_report(lstm_tokens, docs)}")
out.write_text("\n\n" + ("\n\n" + "-" * 40 + "\n\n").join(parts) + "\n\n" + LABEL + "\n", encoding="utf-8")
print(out.read_text())

from google.colab import files
files.download(str(out))

## Export the trained model for your Windows laptop
Run this cell after Part 2 has loaded `docs`. This exports the Markov model as a portable JSON file; no GPU is needed for this main model.

In [ ]:
# Train/export a portable Markov model. Change ORDER to 1, 2, or 3 if desired.
import json
from google.colab import files
from pathlib import Path

EXPORT_ORDER = 2
export_model, export_starts = build_model(docs, EXPORT_ORDER)
export_payload = {
    "format_version": 1,
    "model_type": "word-level Markov chain",
    "order": EXPORT_ORDER,
    "source_files": [p.name for p in (ROOT / "data/raw").glob("*.txt")],
    "word_count": sum(len((ROOT / "data/clean" / p.name).read_text(encoding="utf-8").split()) for p in (ROOT / "data/raw").glob("*.txt")),
    "token_count": sum(len(d) for d in docs),
    "transitions": {"\u241f".join(state): dict(counts) for state, counts in export_model.items()},
    "starts": [list(s) for s in export_starts]
}
export_path = ROOT / "results" / "catbot_model.json"
export_path.write_text(json.dumps(export_payload, ensure_ascii=False), encoding="utf-8")
print(f"Exported {len(export_model):,} states to {export_path}")
files.download(str(export_path))
